# **Uber-Supply-Demand-cleaning**

**Step 1: Import Required Libraries**

In [23]:
import pandas as pd
import numpy as np


pandas is used for data cleaning and analysis.

numpy helps handle missing values.

**Step 2: Load the Dataset**

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Uber Supply Demand/Uber Request Data.csv')


Check the data:

In [26]:
df.head()

,Request id,Pickup point,Driver id,Status,Request timestamp,Drop timestamp
0,619,Airport,1.0,Trip Completed,11/7/2016 11:51,11/7/2016 13:00
1,867,Airport,1.0,Trip Completed,11/7/2016 17:57,11/7/2016 18:47
2,1807,City,1.0,Trip Completed,12/7/2016 9:17,12/7/2016 9:58
3,2532,Airport,1.0,Trip Completed,12/7/2016 21:08,12/7/2016 22:03
4,3112,City,1.0,Trip Completed,13-07-2016 08:33:16,13-07-2016 09:25:47


**Check shape:**

In [27]:
df.shape

(6745, 6)

**Result**

dataset contains:

Rows    : 6745
Columns : 6

**Step 3: Inspect Data Types**

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6745 entries, 0 to 6744
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Request id         6745 non-null   int64  
 1   Pickup point       6745 non-null   object 
 2   Driver id          4095 non-null   float64
 3   Status             6745 non-null   object 
 4   Request timestamp  6745 non-null   object 
 5   Drop timestamp     2831 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 316.3+ KB


**Identifies:**

Missing values


Incorrect data types

Columns requiring cleaning

**Step 4: Check Missing Values**

In [29]:
df.isnull().sum()

,0
Request id,0
Pickup point,0
Driver id,2650
Status,0
Request timestamp,0
Drop timestamp,3914


**Observation:**

These missing values are business-related:

Missing Driver ID → No driver assigned

Missing Drop Timestamp → Trip cancelled or not completed

Therefore:

✅ Keep records

❌ Do not delete rows

**Step 5: Check Duplicate Records**

In [30]:
df.duplicated().sum()

np.int64(0)

Action

In [31]:
df = df.drop_duplicates()

Even though none were found, this step is standard practice.

**Step 6: Standardize Column Names**

**Before:**

['Request id',

 'Pickup point',

 'Driver id',

 'Status',

 'Request timestamp',

 'Drop timestamp']

In [32]:
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
)

**After:**

['Request_id',

 'Pickup_point',

 'Driver_id',

 'Status',

 'Request_timestamp',

 'Drop_timestamp']

**Step 7: Convert Timestamp Columns**

In [33]:
df['Request_timestamp'] = pd.to_datetime(
    df['Request_timestamp'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

df['Drop_timestamp'] = pd.to_datetime(
    df['Drop_timestamp'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

**Allows:**

Hour analysis

Date analysis

Trip duration calculation

**Step 8: Create Date Column**

In [34]:
df['Date'] = df['Request_timestamp'].dt.date

**Step 9: Create Hour Column**

In [35]:
df['Hour'] = df['Request_timestamp'].dt.hour

**Step 10: Create Weekday Column**

In [36]:
df['Weekday'] = (
    df['Request_timestamp']
      .dt.day_name()
)

**Output:**

Monday

Tuesday

Wednesday

Analyze weekday demand.

**Step 11: Create Time Slot Column**

In [37]:
def get_slot(hour):

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 22:
        return "Evening"

    else:
        return "Night"

df["Time_Slot"] = df["Hour"].apply(get_slot)

**Step 12: Create Driver Assigned Flag**

In [38]:
df["Driver_Assigned"] = np.where(
    df["Driver_id"].isnull(),
    "No",
    "Yes"
)

**Step 13: Calculate Trip Duration**

In [39]:
df["Trip_Duration_Minutes"] = (
    df["Drop_timestamp"] -
    df["Request_timestamp"]
).dt.total_seconds()/60

**Step 14: Final Validation**

Check dataset:

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6745 entries, 0 to 6744
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Request_id             6745 non-null   int64         
 1   Pickup_point           6745 non-null   object        
 2   Driver_id              4095 non-null   float64       
 3   Status                 6745 non-null   object        
 4   Request_timestamp      6745 non-null   datetime64[ns]
 5   Drop_timestamp         2831 non-null   datetime64[ns]
 6   Date                   6745 non-null   object        
 7   Hour                   6745 non-null   int32         
 8   Weekday                6745 non-null   object        
 9   Time_Slot              6745 non-null   object        
 10  Driver_Assigned        6745 non-null   object        
 11  Trip_Duration_Minutes  2831 non-null   float64       
dtypes: datetime64[ns](2), float64(2), int32(1), int64(1), object(6

**Check missing values again:**

In [41]:
df.isnull().sum()

,0
Request_id,0
Pickup_point,0
Driver_id,2650
Status,0
Request_timestamp,0
Drop_timestamp,3914
Date,0
Hour,0
Weekday,0
Time_Slot,0


**Expected:**

Driver_id still contains business-valid missing values.

Drop_timestamp still contains business-valid missing values.

**Final Cleaned Dataset Structure**

Request_id

Pickup_point

Driver_id

Status

Request_timestamp

Drop_timestamp

Date

Hour

Weekday

Time_Slot

Driver_Assigned

Trip_Duration_Minutes

**Save Cleaned Data to CSV**

In [42]:
df.to_csv(
    "Uber_Supply_Demand_Cleaned.csv",
    index=False
)